### **🌟 Exercise 3: Scrape Dynamic Content from Rotten Tomatoes**

**Task:**

- Use `Selenium` to navigate to the Rotten Tomatoes Certified Fresh Movies page.
- Extract the HTML content after it’s fully loaded.
- Use `BeautifulSoup` to parse and extract the movie titles, scores, and release dates.

**Instructions**

- Set up `Selenium` WebDriver and navigate to the Rotten Tomatoes page.
- Extract the HTML content using `driver.page_source`.
- Parse the HTML with `BeautifulSoup`.
- Find and extract the desired movie information.
- Print the extracted data.


In [1]:
%pip install google-colab-selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 34.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 89.4 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [google-colab-selenium]nium]


In [41]:
try:
    import google_colab_selenium as gs
except ImportError:
    gs = None

import re
import warnings

import pandas as pd
from bs4 import BeautifulSoup, Tag
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.remote.webdriver import WebDriver
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

In [27]:
URL = "https://www.rottentomatoes.com/browse/movies_in_theaters/critics:certified_fresh"
TILE_SELECTOR = ".discovery-tiles__wrap .flex-container"
MOVIE_FIELDS = {
    "movie_title": '[data-qa="discovery-media-list-item-title"]',
    "critics_score": 'rt-text[slot="criticsScore"]',
    "audience_score": 'rt-text[slot="audienceScore"]',
    "release_date": '[data-qa="discovery-media-list-item-start-date"]',
}

In [59]:
def create_chrome_driver(
    headless: bool = True, user_agent: str | None = None
) -> WebDriver:
    """
    Start a Chrome WebDriver that works both on Google Colab and on a local machine.

    Args:
        headless (bool, optional): Run Chrome without a window. Only applies locally, since
            google-colab-selenium always runs headless on Colab. Defaults to True.

    Returns:
        WebDriver: A ready Chrome driver. The caller is responsible for quitting it,
            ideally by using it in a `with` block.
    """
    options = Options()
    options.add_argument(f"--user-agent={user_agent}")

    if gs is not None:
        return gs.Chrome(options=options)

    if headless:
        options.add_argument("--headless=new")

    return webdriver.Chrome(options=options)

In [60]:
def fetch_rendered_html(
    url: str,
    wait_selector: str | None = None,
    timeout: int = 15,
    headless: bool = True,
    user_agent: str | None = None,
) -> str:
    """
    Load a page in Chrome, optionally wait for an element to appear, and return the rendered HTML.

    Args:
        url (str): The page to load.
        wait_selector (str | None, optional): CSS selector that must be present before the HTML
            is read. None returns the HTML right after the page loads. Defaults to None.
        timeout (int, optional): Seconds to wait for wait_selector. Defaults to 15.
        headless (bool, optional): Run Chrome without a window (local runs only). Defaults to True.

    Returns:
        str: The page's HTML after JavaScript has rendered it.

    Raises:
        TimeoutException: If wait_selector doesn't appear within timeout seconds.
    """
    with create_chrome_driver(headless, user_agent) as driver:
        driver.get(url)

        if wait_selector is not None:
            try:
                WebDriverWait(driver, timeout).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, wait_selector))
                )
            except TimeoutException as error:
                raise TimeoutException(
                    f"'{wait_selector}' did not appear on {url} within {timeout}s. "
                    "Check the selector in DevTools or increase the timeout."
                ) from error

        return driver.page_source

In [15]:
def extract_value(
    element: Tag,
    selector: str | None = None,
    attribute: str | None = None,
    separator: str = " ",
) -> str | None:
    """
    Return the text or an attribute of the first match for a CSS selector inside an element.

    Args:
        element (Tag): The element to search inside.
        selector (str | None, optional): CSS selector relative to element. None uses element itself. Defaults to None.
        attribute (str | None, optional): Attribute to read (e.g. 'href'). None returns the text. Defaults to None.
        separator (str, optional): String placed between the text of nested elements. Defaults to " ".

    Returns:
        str | None: The stripped text or attribute value, or None if nothing matched.
    """
    target = element if selector is None else element.select_one(selector)

    if target is None:
        return None

    if attribute is not None:
        return target.get(attribute)

    return target.get_text(separator=separator, strip=True)

In [ ]:
def scrape_records(
    html: str,
    container_selector: str,
    fields: dict[str, str | tuple[str, str]],
    parser: str = "html.parser",
) -> list[dict[str, str | None]]:
    """
    Parse HTML into one dict per repeated container (a card, row, or list item), one key per field.

    Args:
        html (str): The page HTML, e.g. from fetch_rendered_html.
        container_selector (str): CSS selector matching one element per record, e.g. '.wr-day'.
        fields (dict[str, str | tuple[str, str]]): Column name -> CSS selector inside the container
            for its text, or -> (selector, attribute) to read an attribute such as 'href'.
        parser (str, optional): BeautifulSoup parser to use. Defaults to "html.parser".

    Returns:
        list[dict[str, str | None]]: One dict of raw string values per container, None where a field is missing.

    Raises:
        ValueError: If container_selector matches nothing on the page.
    """
    containers = BeautifulSoup(html, parser).select(container_selector)

    if not containers:
        raise ValueError(
            f"No elements match '{container_selector}'. Check the selector in DevTools."
        )

    records = []
    for container in containers:
        record = {}
        for column, spec in fields.items():
            selector, attribute = spec if isinstance(spec, tuple) else (spec, None)
            record[column] = extract_value(container, selector, attribute)
        records.append(record)

    for column in fields:
        if all(record[column] is None for record in records):
            warnings.warn(
                f"Field '{column}' is None in every record. Its selector may be wrong."
            )

    return records

In [24]:
html = fetch_rendered_html(URL, wait_selector=TILE_SELECTOR)

<IPython.core.display.Javascript object>

In [29]:
movies = scrape_records(html, TILE_SELECTOR, MOVIE_FIELDS)

In [33]:
title_width = max(len(movie["movie_title"] or "") for movie in movies)

print(f"{'Title':<{title_width}}  {'Critics':>7}  {'Audience':>8}  Release")
print("-" * (title_width + 46))

for movie in movies:
    print(
        f"{movie['movie_title'] or '—':<{title_width}}  "
        f"{movie['critics_score'] or '—':>7}  "
        f"{movie['audience_score'] or '—':>8}  "
        f"{movie['release_date'] or '—'}"
    )

Title                                   Critics  Audience  Release
------------------------------------------------------------------------------------
Scream                                      78%       80%  Re-releasing Oct 30, 2026
V for Vendetta                              73%       90%  Re-releasing Nov 01, 2026
Sense and Sensibility                       77%       88%  Opened Oct 16, 2026
Resident Evil                               96%       90%  Opened Sep 18, 2026
Heart of the Beast                          84%       95%  Opened Sep 25, 2026
Forgotten Island                            95%       94%  Opened Sep 25, 2026
Your Mother Your Mother Your Mother         96%       93%  Opened Oct 09, 2026
Spider-Man: Brand New Day                   90%       97%  Opened Jul 31, 2026
The Odyssey                                 94%       96%  Opened Jul 17, 2026
Musk                                        96%       95%  Opened Oct 09, 2026
NAZA                                       100

### **🌟 Exercise 4: Scrape and Categorize News Articles from a JavaScript-Enabled News Site**

**Task:**

- Visit this [website](https://www.bbc.com/innovation/technology).
- Scrape news article titles and their publication dates.
- Categorize articles based on their publication month.

**Instructions:**

- Use `Selenium` to navigate to a specific news section on the website.
- Extract and parse the HTML content that is dynamically loaded via JavaScript.
- Using `BeautifulSoup`, extract news article titles and publication dates.
- Categorize articles by their publication month (e.g., `‘January’`, `‘February’`, etc.).
- Print the categorized lists of articles.


In [ ]:
NEWS_URL = "https://www.bbc.com/technology"

ARTICLE_SELECTOR = (
    '[data-testid$="-card"]:has([data-testid="card-metadata-lastupdated"])'
)
ARTICLE_FIELDS = {
    "title": '[data-testid="card-headline"]',
    "published": '[data-testid="card-metadata-lastupdated"]',
}

#### Choosing what to scrape

The page has about 65 article cards, but only the 15 in the **More Tech** section show a publication date. The feature, video and "In case you missed it" cards have none. Since the task is to categorize by publication month, the container selector keeps only cards that have a date.

The dates are **relative** ("Just now", "3 hrs ago", "2 days ago"), so they're converted into timestamps by counting back from the moment of scraping.


In [35]:
news_html = fetch_rendered_html(NEWS_URL, wait_selector=ARTICLE_SELECTOR)

<IPython.core.display.Javascript object>

In [37]:
SCRAPED_AT = pd.Timestamp.now(tz="Europe/London").tz_localize(None)

In [38]:
article_rows = scrape_records(news_html, ARTICLE_SELECTOR, ARTICLE_FIELDS)
print(len(article_rows))

15


In [42]:
def to_publication_time(
    published: str | None, scraped_at: pd.Timestamp
) -> pd.Timestamp:
    """
    Convert BBC's card date text ('Just now', '3 hrs ago', '2 days ago' or '12 Oct 2026') into a timestamp.

    Args:
        published (str | None): The scraped date text from one article card.
        scraped_at (pd.Timestamp): When the page was scraped, in UK time, used as 'now' for relative dates.

    Returns:
        pd.Timestamp: The approximate publication time, or NaT if the text is missing or unrecognized.
    """
    if published is None:
        return pd.NaT

    if published.strip().lower() == "just now":
        return scraped_at

    relative = re.fullmatch(r"(\d+)\s+(min|hr|day)s?\s+ago", published.strip())
    if relative:
        amount, unit = int(relative.group(1)), relative.group(2)
        unit_names = {"min": "minutes", "hr": "hours", "day": "days"}
        return scraped_at - pd.Timedelta(**{unit_names[unit]: amount})

    return pd.to_datetime(published, format="%d %b %Y", errors="coerce")


In [43]:
articles = pd.DataFrame(article_rows)
articles["published_at"] = [
    to_publication_time(text, SCRAPED_AT) for text in articles["published"]
]

In [44]:
articles["month"] = articles["published_at"].dt.month_name()

In [45]:
articles["month"].isna().sum()

np.int64(0)

In [46]:
for month, group in articles.groupby("month", sort=False):
    print(f"{month} ({len(group)} articles)")
    for title in group["title"]:
        print(f"  - {title}")
    print()

October (15 articles)
  - Ofcom outlines age check methods ahead of social media ban for under-16s
  - New data centres must bring 'local benefits'
  - 'Super offensive' - we love how we look but AI tried to 'fix' our visible differences
  - Microsoft profits soar days after announcing thousands of layoffs
  - Elon Musk is a 'real genius', Putin says after satellite talks with SpaceX boss
  - Babydoll singer says life will 'never be the same again' after UK number one
  - Celebrity Traitors' Joe Lycett's tribute to viral TikTok nan
  - Jo Malone donates £100k for cancer-detecting 'nose'
  - Study to investigate harmful adult-child messaging
  - A bust-up between mathematicians is fuelling fears that AI could undermine science
  - 'We got a day off work thanks to Erling Haaland'
  - Ex-Britain First leader arrested in HMO attacks investigation
  - These Storm Boelo images were not AI - but some people thought they were
  - Girl asks council to 'fight' in data centre row
  - 'Maxxing' se

### **🌟 Exercise 5: Scrape and Analyze Weather Data from a JavaScript-Enabled Weather Website**

**Task:**

- Visit this [website](https://www.accuweather.com/en/us/los-angeles-ca/90012/weather-forecast/348108).
- Scrape weather forecast data including temperature, condition, and humidity.
- Analyze the data to find the average temperature and most common weather condition.

**Instructions:**

- Use `Selenium` to navigate to the weather forecast page of a specific city.
- Extract and parse the HTML content, focusing on dynamically loaded weather data.
- Using `BeautifulSoup`, extract relevant weather information like temperature, condition (sunny, cloudy, etc.), and humidity.
- Calculate the average temperature and identify the most common weather condition.
- Print the analysis results.


In [71]:
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36"
)

WEATHER_URL = "https://www.accuweather.com/en/us/attica/30607/weather-forecast/2139413"
CURRENT_URL = "https://www.accuweather.com/en/us/attica/30607/current-weather/2139413"

DAY_SELECTOR = ".daily-list-item"
DAY_FIELDS = {
    "day": ".date .day",
    "date": ".date p:nth-of-type(2)",
    "high_temp": ".temp-hi",
    "low_temp": ".temp-lo",
    "condition": ".phrase > p",
}

HUMIDITY_ITEM_SELECTOR = ".detail-item"
HUMIDITY_SELECTOR = '.detail-item:-soup-contains("Humidity") > div:nth-of-type(2)'

#### Approach

- **Blocked at first:** headless Chrome identifies itself as `HeadlessChrome`, and AccuWeather returned "Access Denied". Sending a normal Chrome user-agent loads the real page.
- **Two pages:** the forecast page has a 10-day list (one `.daily-list-item` per day, with high, low and condition) but no visible humidity. Humidity is shown on the current-weather page, so it's scraped from there as a single current value.
- **Condition:** each day's _daytime_ condition (`.phrase > p`). The night condition is in a separate `span.night` inside the same block, and the `>` (direct child) leaves it out.
- **Humidity selector:** the detail rows have no unique classes, only a label and a value (`<div>Humidity</div><div>85%</div>`). `:-soup-contains("Humidity")` picks the row whose text contains "Humidity", and `div:nth-of-type(2)` takes its value. `:-soup-contains` works in BeautifulSoup only, not in browser CSS, so the Selenium wait uses plain `.detail-item`.
- **Average temperature:** each day's average is approximated as the midpoint of its high and low, `(high + low) / 2`. The page doesn't give hourly temperatures for a true daily mean.


In [ ]:
forecast_html = fetch_rendered_html(
    WEATHER_URL, wait_selector=DAY_SELECTOR, user_agent=USER_AGENT
)

<IPython.core.display.Javascript object>

In [72]:
current_html = fetch_rendered_html(
    CURRENT_URL, wait_selector=HUMIDITY_ITEM_SELECTOR, user_agent=USER_AGENT
)

<IPython.core.display.Javascript object>

In [73]:
day_rows = scrape_records(forecast_html, DAY_SELECTOR, DAY_FIELDS)
print(len(day_rows))
print(day_rows[0])

10
{'day': 'Today', 'date': '10/29', 'high_temp': '78°', 'low_temp': '49°', 'condition': 'Partly sunny and pleasant'}


In [ ]:
current_humidity = extract_value(
    BeautifulSoup(current_html, "html.parser"), HUMIDITY_SELECTOR
)
print(current_humidity)

85%


In [86]:
def to_number(text: str | None) -> float | None:
    """
    Pull the first number out of scraped text such as '26°' or '85%'.

    Args:
        text (str | None): The scraped text.

    Returns:
        float | None: The number found, or None if the text is missing or has no number.
    """
    if text is None:
        return None

    match = re.search(r"-?\d+(?:\.\d+)?", text)

    return float(match.group()) if match else None

In [78]:
def fahrenheit_to_celsius(fahrenheit: float | pd.Series) -> float | pd.Series:
    """
    Convert temperatures from Fahrenheit to Celsius.

    Args:
        fahrenheit (float | pd.Series): One temperature or a Series of temperatures in °F.

    Returns:
        float | pd.Series: The same temperature(s) in °C.
    """
    return (fahrenheit - 32) * 5 / 9

In [ ]:
forecast = pd.DataFrame(day_rows)

for column in ["high_temp", "low_temp"]:
    forecast[column] = fahrenheit_to_celsius(forecast[column].map(to_number))

In [89]:
forecast["avg_temp"] = (forecast["high_temp"] + forecast["low_temp"]) / 2

In [90]:
temp_averages = forecast[["high_temp", "low_temp", "avg_temp"]].mean().round(1)

In [91]:
condition_counts = forecast["condition"].value_counts()
most_common_condition = condition_counts.idxmax()

In [ ]:
print(
    f"{len(forecast)}-day forecast ({forecast['date'].iloc[0]} to {forecast['date'].iloc[-1]})"
)
print(f"Average high: {temp_averages['high_temp']}°")
print(f"Average low: {temp_averages['low_temp']}°")
print(f"Average daily temperature: {temp_averages['avg_temp']}°")
print(
    f"Most common condition: {most_common_condition} ({condition_counts.max()} of {len(forecast)} days)"
)
print(f"Current humidity: {current_humidity}")

10-day forecast (10/29 to 11/7)
Average high: 22.2°
Average low: 12.4°
Average daily temperature: 17.3°
Most common condition: Cloudy (2 of 10 days)
Current humidity: 85%
